# ЛР2 — аудиторный demo-notebook

**Тема:** Data pipeline: от сырой таблицы до честного эксперимента  
**Формат:** интерактивный разбор на занятии, не решение домашней ЛР2  
**Данные:** маленькая синтетическая таблица, встроенная прямо в notebook

> **Центральный вопрос:** как подготовить данные так, чтобы результат модели можно было считать честным и воспроизводимым?

На занятии мы используем другой датасет, другие размеры split и другой `random_state`, чем в домашней ЛР2. Идеи те же, ответы — не те же.

> **Как работаем с demo.** Этот notebook открыт у всех участников занятия. Код уже находится в ячейках — перепечатывать его с экрана не нужно. В местах с вопросами сначала зафиксируйте прогноз, затем запускайте ячейку и сравнивайте результат с ожиданием.


## 0. Окружение

После ЛР1 работа с Jupyter уже знакома. Здесь достаточно напоминания: перед демонстрацией полезно выполнить **Restart Kernel → Run All**.

GPU не нужен.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, roc_auc_score

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)

DEMO_SEED = 1

# 1. Сначала — обычная таблица

Это **синтетические** данные о 24 условных студентах. Они нужны только для демонстрации pipeline.

До запуска следующих ячеек сначала ответьте:

- что здесь является строкой / объектом;
- какие столбцы похожи на числовые признаки;
- какой столбец — target;
- есть ли столбец, который хранится числом, но по смыслу является категорией.

In [ ]:
demo = pd.DataFrame({
    "StudyHours": [1.5, 2.0, 3.0, 4.0, 5.0, 6.0, np.nan, 8.0,
                   2.5, 3.5, 4.5, 5.5, 6.5, 7.5, 1.0, 2.8,
                   3.8, 4.8, 5.8, np.nan, 7.8, 8.5, 4.2, 5.2],
    "Attendance": [52, 60, 68, 75, 80, 88, 92, 95,
                   58, 65, np.nan, 78, 84, 90, 50, 63,
                   70, 76, 82, 87, 93, 97, 74, 79],
    "Group": [1, 1, 2, 2, 3, 1, 2, 3, 1, 2, 3, 1,
              2, 3, 1, 2, 3, 1, 2, 3, 1, 2, 3, 1],
    "Format": ["online", "online", "hybrid", "onsite", "onsite", "hybrid",
               "onsite", "onsite", "online", "hybrid", "hybrid", "onsite",
               "online", "onsite", "online", "hybrid", None, "onsite",
               "hybrid", "online", "onsite", "onsite", "hybrid", "online"],
    "Passed": [0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1,
               1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1],
})

display(demo.head(8))
print("shape:", demo.shape)

## 1.1. Первичная диагностика: сначала выясняем, что у нас есть

Не нужно сразу «чистить данные». Сначала задаём несколько конкретных вопросов.

**Перед запуском:** где ожидаются пропуски? Сбалансирован ли target идеально?

In [ ]:
print("Типы столбцов:")
display(demo.dtypes.to_frame("dtype"))

print("Пропуски:")
display(demo.isna().sum().to_frame("missing"))

print("Target:")
display(demo["Passed"].value_counts().sort_index().to_frame("count"))
print("Доля Passed=1:", round(demo["Passed"].mean(), 3))

### Важный вопрос: `dtype` — это ещё не роль признака

`Group` хранится как целое число, но значения 1, 2, 3 обозначают **названия групп**, а не измеренную величину.

Поэтому в этом demo мы будем трактовать:

- `StudyHours`, `Attendance` — как числовые признаки;
- `Group`, `Format` — как категориальные;
- `Passed` — как target.

Это решение определяется смыслом данных, а не только тем, что показывает `dtypes`.

# 2. Мини-EDA: график отвечает на вопрос

Вместо каталога графиков зададим один вопрос:

> отличаются ли учебные часы у двух классов target?

До запуска сначала спрогнозируйте, что ожидаете увидеть.

In [ ]:
plt.figure(figsize=(6, 3))
demo.loc[demo["Passed"] == 0, "StudyHours"].dropna().hist(
    bins=6, alpha=0.65, label="Passed=0"
)
demo.loc[demo["Passed"] == 1, "StudyHours"].dropna().hist(
    bins=6, alpha=0.65, label="Passed=1"
)
plt.xlabel("StudyHours")
plt.ylabel("Количество")
plt.title("StudyHours по классам target")
plt.legend()
plt.tight_layout()
plt.show()

**Обсуждение**

- Видна ли связь?
- Достаточно ли одного графика, чтобы говорить о причинности? **Нет.**
- Нужен ли этот график для принятия решения о pipeline? Он помогает понять данные, но сам preprocessing ещё не определяет.

# 3. Split: сначала разделяем данные, потом обучаем preprocessing

В demo используем **60% / 20% / 20%** и `DEMO_SEED = 1` — специально не те параметры, что в домашней ЛР2.

`stratify` помогает сохранить близкие доли классов в каждой части.

До запуска ответьте: **какая часть должна оставаться «запечатанной» до финальной проверки?**

In [ ]:
TARGET = "Passed"
NUMERIC_FEATURES = ["StudyHours", "Attendance"]
CATEGORICAL_FEATURES = ["Group", "Format"]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

X = demo[FEATURES].copy()
y = demo[TARGET].copy()
all_idx = np.arange(len(demo))

train_val_idx, test_idx = train_test_split(
    all_idx,
    test_size=0.20,
    random_state=DEMO_SEED,
    stratify=y,
)

train_idx, val_idx = train_test_split(
    train_val_idx,
    test_size=0.25,  # 25% от оставшихся 80% = 20% всей выборки
    random_state=DEMO_SEED,
    stratify=y.iloc[train_val_idx],
)

X_train = X.iloc[train_idx].copy()
X_val = X.iloc[val_idx].copy()
X_test = X.iloc[test_idx].copy()
y_train = y.iloc[train_idx].copy()
y_val = y.iloc[val_idx].copy()
y_test = y.iloc[test_idx].copy()

split_summary = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "n": [len(y_train), len(y_val), len(y_test)],
    "positive_rate": [y_train.mean(), y_val.mean(), y_test.mean()],
})
display(split_summary)

assert set(train_idx).isdisjoint(val_idx)
assert set(train_idx).isdisjoint(test_idx)
assert set(val_idx).isdisjoint(test_idx)

# 4. Data leakage: правильная формула на неправильных данных

В `StudyHours` есть пропуски. Допустим, мы хотим заполнить их медианой.

Вопрос не только **«какую медиану?»**, но и **«по какой части данных она посчитана?»**.

In [ ]:
full_median = demo["StudyHours"].median()
train_median = X_train["StudyHours"].median()

print("Медиана по ВСЕМ данным:", full_median)
print("Медиана только по TRAIN:", train_median)

**Ключевой вывод**

Если медиану посчитать по всей таблице, validation/test уже повлияли на правило подготовки train.
Это и есть утечка информации.

Даже если две медианы получились близкими, методика от этого не становится корректной.

> `fit` — **узнать параметры преобразования по train**.  
> `transform` — **применить уже выученное правило к новым данным**.

# 5. Разные столбцы — разные ветки preprocessing

Числовые признаки:

```text
StudyHours, Attendance
        ↓
median imputation
        ↓
StandardScaler
```

Категориальные признаки:

```text
Group, Format
        ↓
most-frequent imputation
        ↓
OneHotEncoder
```

`ColumnTransformer` объединяет эти две ветки в один преобразователь.

In [ ]:
def make_demo_preprocessor():
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])

    return ColumnTransformer([
        ("num", numeric_pipeline, NUMERIC_FEATURES),
        ("cat", categorical_pipeline, CATEGORICAL_FEATURES),
    ])

preprocessor = make_demo_preprocessor()

X_train_prepared = preprocessor.fit_transform(X_train)
X_val_prepared = preprocessor.transform(X_val)

print("raw train shape:", X_train.shape)
print("prepared train shape:", X_train_prepared.shape)
print("prepared val shape:", X_val_prepared.shape)
print("feature names:")
print(preprocessor.get_feature_names_out())

### Почему столбцов стало больше?

Не запускайте ничего дополнительно — сначала объясните словами.

`OneHotEncoder` превращает одну категориальную колонку в несколько индикаторных колонок.
Поэтому число строк сохраняется, а число признаков может увеличиться.

Отдельно обратите внимание: `Group` исходно был `int`, но мы всё равно отправили его в категориальную ветку — потому что таков **смысл** признака.

# 6. `Pipeline`: сырая строка на входе, предсказание на выходе

Теперь объединим preprocessing и модель в один объект.

На занятии модель здесь не является главным героем. Наша цель — увидеть, что `Pipeline` сам:

1. обучает preprocessing только внутри `fit(X_train, y_train)`;
2. применяет тот же preprocessing внутри `predict(X_val)`;
3. не требует вручную синхронизировать преобразования между split.

In [ ]:
demo_pipeline = Pipeline([
    ("preprocessor", make_demo_preprocessor()),
    ("model", LogisticRegression(max_iter=1000, random_state=DEMO_SEED)),
])

demo_pipeline.fit(X_train, y_train)
val_pred = demo_pipeline.predict(X_val)

preview = X_val.copy()
preview["y_true"] = y_val.to_numpy()
preview["y_pred"] = val_pred

display(preview)

# 7. Вероятность и threshold: модель не обязана решать только при 0.5

`predict_proba` возвращает оценку вероятности положительного класса. Если изменить threshold, изменится и баланс между ошибками.

**До запуска:** что, скорее всего, произойдёт с recall положительного класса, если снизить threshold с `0.7` до `0.3`?


In [ ]:
val_proba = demo_pipeline.predict_proba(X_val)[:, 1]

rows = []
for threshold in [0.3, 0.5, 0.7]:
    pred = (val_proba >= threshold).astype(int)
    rows.append({
        'threshold': threshold,
        'precision': precision_score(y_val, pred, zero_division=0),
        'recall': recall_score(y_val, pred, zero_division=0),
    })

display(pd.DataFrame(rows).round(3))
print('ROC-AUC:', round(roc_auc_score(y_val, val_proba), 3))


На такой маленькой demo-выборке числа скачут, поэтому здесь важна не конкретная величина. Важен принцип: **threshold выбирает рабочую точку precision/recall, а ROC-AUC оценивает ранжирование вероятностей сразу по множеству возможных порогов**. В самостоятельной части это проверяется уже на Titanic validation.


# 8. Мини-checkpoint

Сначала ответьте **без кода**.

### Ситуация A
Мы посчитали среднее `Attendance` по всей таблице, а затем сделали split и заполнили пропуски этим значением.

**Корректно или leakage? Почему?**

### Ситуация B
`StandardScaler` сделал `fit` на `X_train`, после чего `transform` применён к `X_val` и `X_test`.

**Корректно или leakage?**

### Ситуация C
После one-hot encoding число столбцов увеличилось с 4 до 8.

**Это обязательно ошибка? Почему?**

### Ситуация D
`Group` имеет dtype `int64`.

**Должны ли мы автоматически масштабировать его как обычное число?**

# 9. Переход к самостоятельной работе этой практики

На занятии мы разобрали **логику** pipeline на маленькой синтетической таблице.

Теперь откройте `student.ipynb` — единую сдаваемую работу темы. В ней вы самостоятельно примените эту логику к Titanic:

- проведёте диагностику реального CSV;
- построите несколько осмысленных визуализаций;
- сделаете фиксированный `train/validation/test` split;
- соберёте `ColumnTransformer`;
- сравните Dummy baseline и Logistic Regression;
- посчитаете accuracy, F1 и confusion matrix;
- посмотрите конкретные ошибки;
- сохраните подготовленные данные для ЛР3.

Главный вопрос перед любой строкой preprocessing:

> **Это преобразование что-то «узнаёт» из данных? Если да — по какой части данных оно имеет право это узнавать?**